# 第82章 随机森林

理解随机森林如何通过样本和特征随机化集成多棵树，并使用袋外评估与置换重要性诊断模型。


## 先解决一个小问题

围绕“随机森林”完成一个可验证的小型建模实验：先明确输入和目标，再比较方法带来的变化。理解随机森林如何通过样本和特征随机化集成多棵树，并使用袋外评估与置换重要性诊断模型。


## 这章为什么先学

这是“机器学习”建模主线中的第 82 章，重点放在“随机森林”对应的一个具体决策，而不是重复完整流程。


## 开始前确认

- 能够使用 pandas 读取、筛选和汇总数据
- 理解训练集、测试集和基本统计指标
- 本章会进一步练习：训练 RandomForestClassifier、理解 bootstrap 与特征子采样、使用 OOB 分数


## 做完要留下什么

完成一份围绕“随机森林”的可运行实验：包含数据准备、方法执行、指标或图表证据，以及一句有边界的结论。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 训练 RandomForestClassifier
- 理解 bootstrap 与特征子采样
- 使用 OOB 分数
- 比较内置重要性与置换重要性


## 核心概念

- 多棵低相关树平均可降低方差
- n_estimators 主要影响稳定性和计算量
- OOB 使用每棵树未抽中的样本评估
- 置换重要性衡量打乱特征后的性能下降


## 示例 1：随机森林与 OOB

在乳腺癌数据上比较测试与袋外分数。


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
data=load_breast_cancer(as_frame=True); X,y=data.data,data.target
X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y,random_state=82)
forest=RandomForestClassifier(n_estimators=300,min_samples_leaf=3,oob_score=True,n_jobs=-1,random_state=82).fit(X_train,y_train)
print('OOB/测试:',round(forest.oob_score_,3),round(forest.score(X_test,y_test),3))


## 示例 2：两种特征重要性

置换重要性在测试集测量，更直接反映泛化性能依赖。


In [ ]:
import pandas as pd
from sklearn.inspection import permutation_importance
impurity=pd.Series(forest.feature_importances_,index=X.columns,name='impurity')
perm=permutation_importance(forest,X_test,y_test,n_repeats=10,random_state=82,n_jobs=-1)
compare=pd.concat([impurity,pd.Series(perm.importances_mean,index=X.columns,name='permutation')],axis=1)
display(compare.sort_values('permutation',ascending=False).head(10).round(4))


## 建模流程提醒

1. **定义问题**：写清楚样本粒度、预测时点、目标变量和业务代价。
2. **建立基线**：先用均值、规则或 Dummy 模型得到最低可接受结果。
3. **准备数据**：只用预测时点可获得的信息，避免目标泄漏和时间穿越。
4. **训练与验证**：在训练/验证数据上选择方案，测试集只用于最终估计泛化表现。
5. **评价与解释**：同时看总体指标、错误切片和结果边界，不能只报一个分数。


## 常见误区

- 认为更多树一定解决所有过拟合
- 高基数特征下只看 impurity importance
- 用测试集反复筛特征
- 忽略森林比单树更难解释且占用更多资源


## 综合练习

1. 比较 50、150、300 棵树
2. 记录 OOB 和测试得分
3. 观察分数是否趋于稳定

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“比较 50、150、300 棵树”。
2. **独立完成**：不复制示例代码，完成“记录 OOB 和测试得分”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“观察分数是否趋于稳定”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
forest_rows=[]
for n in [50,150,300]:
    m=RandomForestClassifier(n_estimators=n,min_samples_leaf=3,oob_score=True,n_jobs=-1,random_state=82).fit(X_train,y_train)
    forest_rows.append([n,m.oob_score_,m.score(X_test,y_test)])
practice_result=pd.DataFrame(forest_rows,columns=['trees','oob','test'])
display(practice_result.round(3))

# 自检
assert len(practice_result) == 3
assert practice_result[['oob','test']].le(1).all().all()


## 本章小结

理解随机森林如何通过样本和特征随机化集成多棵树，并使用袋外评估与置换重要性诊断模型。


### 你已经掌握

- 训练 RandomForestClassifier
- 理解 bootstrap 与特征子采样
- 使用 OOB 分数
- 比较内置重要性与置换重要性


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 随机森林与 OOB | 在乳腺癌数据上比较测试与袋外分数。 | `forest.score()`、`.fit()` |
| 两种特征重要性 | 置换重要性在测试集测量，更直接反映泛化性能依赖。 | `pd.Series()`、`pd.concat()`、`compare.sort_values()`、`.head()` |


### 需要注意

- 认为更多树一定解决所有过拟合
- 高基数特征下只看 impurity importance
- 用测试集反复筛特征
- 忽略森林比单树更难解释且占用更多资源


### 完成检查

- [ ] 能够训练 RandomForestClassifier
- [ ] 能够理解 bootstrap 与特征子采样
- [ ] 能够使用 OOB 分数
- [ ] 能够比较内置重要性与置换重要性


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
